In [33]:
import pandas as pd
import numpy as np
from sklearn.metrics import mutual_info_score

In [5]:
import requests

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
output_name = "course_lead_scoring_2026.csv"

# Send a GET request to the URL
response = requests.get(url)

# Check if the request was successful
if response.status_code == 200:
    # Open a local file in write-binary mode and save the content
    with open(output_name, "wb") as file:
        file.write(response.content)
    print(f"File successfully downloaded and saved as {output_name}")
else:
    print(f"Failed to download file. Status code: {response.status_code}")

File successfully downloaded and saved as course_lead_scoring_2026.csv


In [7]:
df = pd.read_csv("course_lead_scoring_2026.csv")

#check null values

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               5000 non-null   object 
 1   industry                  5000 non-null   object 
 2   employment_status         5000 non-null   object 
 3   location                  5000 non-null   object 
 4   annual_income             5000 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                5000 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [13]:
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].fillna("NA")

In [141]:
numerical = list(df.dtypes[df.dtypes != 'object'].index)

for c in numerical:
    df[c] = df[c].fillna(0)

In [142]:
df["industry"].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [143]:
#create correlation matrix
df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


Question 1
What is the most frequent observation (mode) for the column industry?

Answers: 
technology


#Setting up the validation framework

In [144]:
from sklearn.model_selection import train_test_split

In [145]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [146]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [147]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [148]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [149]:
q3_Dict = dict()
biggest = 0.00000001
for col in ["industry","location","lead_source","employment_status"]:
    mi_score = round(mutual_info_score(df_full_train.converted, df_full_train[col]),3)
    q3_Dict[col] = mi_score
    if mi_score > biggest: 
        biggest = mi_score

In [150]:
q3_Dict

{'industry': 0.002,
 'location': 0.001,
 'lead_source': 0.033,
 'employment_status': 0.022}

In [151]:
print(biggest)

0.033


In [152]:
numerical_iv = list(df_train.dtypes[df_train.dtypes != 'object'].index)

In [153]:
numerical

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'converted']

In [155]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical_columns + numerical_iv].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

In [156]:
X_train.shape

(3000, 28)

In [158]:
val_dict = df_val[categorical_columns + numerical_iv].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [159]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [160]:
y_pred = model.predict_proba(X_val)[:, 1]
churn_decision = (y_pred >= 0.5)
round((y_val == churn_decision).mean(),2)

np.float64(0.65)

In [161]:
accuracy_all_features = (y_val == churn_decision).mean()

In [162]:
accuracy_all_features

np.float64(0.651)

In [163]:
numerical_iv = list(df_train.dtypes[df_train.dtypes != 'object'].index)

In [164]:
numerical_iv = list(df_train.dtypes[df_train.dtypes != 'object'].index)
numerical_iv.remove("lead_score")
one = numerical_iv
numerical_iv = list(df_train.dtypes[df_train.dtypes != 'object'].index)
numerical_iv.remove("number_of_courses_viewed")
two = numerical_iv
numerical_iv = list(df_train.dtypes[df_train.dtypes != 'object'].index)
numerical_iv.remove("interaction_count")
three = numerical_iv
numerical_iv = list(df_train.dtypes[df_train.dtypes != 'object'].index)

In [180]:
for c_val in [0.000001, 0.00001, 0.0001, 0.001]:
    train_dict = df_train[categorical_columns + numerical_iv].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)
    val_dict = df_val[categorical_columns + numerical_iv].to_dict(orient='records')
    X_val = dv.transform(val_dict)
    model = LogisticRegression(solver='liblinear', C=c_val, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]
    churn_decision = (y_pred >= 0.5)
    accuracy = round((y_val == churn_decision).mean(),3)
    print(f"accuracy for model with {c_val}")
    print(accuracy)
    

accuracy for model with 1e-06
0.59
accuracy for model with 1e-05
0.59
accuracy for model with 0.0001
0.59
accuracy for model with 0.001
0.648
